### Orders cleaning note
- `order_estimated_delivery_date` contains a time component of `00:00:00` for all rows.
- The time portion carries no useful information.
- Convert the field to date-only format.


I thought there wasn't much to check but I should be more observant
Chat pointed out that there could be data that doesn't make sense like if the order status says delivered then there shouldn't be empty delivery times

We should also check if the dates seem chronological as in 
order must be purchased -> approved -> shipped > delivered

The estimated_date_delivered is what the company suspects the package should be delivered by

In [1]:
import pandas as pd
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
orders.head(5)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [2]:
orders.duplicated().sum()

np.int64(0)

In [3]:
orders.isna().sum()
# you were thinking each row with null info means we should get rid of the row but this shows
# that some orders were never approved, some never shipped out, and some never delivered

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [4]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [5]:
orders["order_id"].duplicated().sum()
orders["customer_id"].duplicated().sum()



np.int64(0)

In [6]:
print(orders["order_estimated_delivery_date"].str.contains("00:00:00", na=False).sum())
orders["order_estimated_delivery_date"].notna().sum()

# Because each row contains 00:00:00 as the time we can rid of that and
# make estimated_delivery_data as a date object instead of datetime

99441


np.int64(99441)

In [7]:
# if order status = delivered the order time should not be empty

orders[orders["order_status"] == "delivered"]["order_delivered_customer_date"].isna().sum()

np.int64(8)

In [8]:
orders[(orders["order_status"] == "delivered") & (orders["order_delivered_customer_date"].isna())]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


In [11]:
date_columns = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]
for column in date_columns:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")
orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [18]:
orders["flag_invalid_purchase_approval"] = (orders["order_purchase_timestamp"] > orders["order_approved_at"])
print(orders["flag_invalid_purchase_approval"].sum())

0


In [ ]:
orders["flag_invalid_approval_carrier"] = (
    orders["order_approved_at"] > orders["order_delivered_carrier_date"]
)
print(orders["flag_invalid_approval_carrier"].sum())

# Found 1,359 orders where the recorded carrier handoff precedes approval.
# This may reflect the actual process or a delay in recording events; the cause is unconfirmed.
# Original timestamps were retained.



1359


In [ ]:
orders["flag_invalid_carrier_delivery"] = (
    orders["order_delivered_carrier_date"]
    > orders["order_delivered_customer_date"]
)
print(orders["flag_invalid_carrier_delivery"].sum())

# theres 23 rows where the carrier delivery date is after the customer delivery date which is impossible

23


In [22]:
orders["flag_invalid_purchase_estimated"] = (
    orders["order_purchase_timestamp"]
    > orders["order_estimated_delivery_date"]
)
print(orders["flag_invalid_purchase_estimated"].sum())

0


In [ ]:
orders["flag_missing_delivery_date"] = (
    (orders["order_status"] == "delivered")
    & orders["order_delivered_customer_date"].isna()
)
print(orders["flag_missing_delivery_date"].sum())

# 8 rows have missing delivery dates despite having a status of delivered

8


In [ ]:
orders["flag_status_delivery_mismatch"] = (
    (orders["order_status"] != "delivered")
    & orders["order_delivered_customer_date"].notna()
)
print(orders["flag_status_delivery_mismatch"].sum())

orders.loc[
    orders["flag_status_delivery_mismatch"],
    ["order_id", "order_status", "order_delivered_customer_date"]
]

### Order status and delivery date mismatch
# Found 6 orders marked `canceled` with a recorded customer delivery timestamp.
# The available fields do not establish whether the status or timestamp is incorrect.
# Retained the original values and flagged these rows using `flag_status_delivery_mismatch`.

6


,order_id,order_status,order_delivered_customer_date
2921,1950d777989f6a877539f53795b4c3c3,canceled,2018-03-21 22:03:51
8791,dabf2b0e35b423f94618bf965fcb7514,canceled,2016-10-16 14:36:59
58266,770d331c84e5b214bd9dc70a10b829d0,canceled,2016-10-14 15:07:11
59332,8beb59392e21af5eb9547ae1a9938d06,canceled,2016-10-19 18:47:43
92636,65d1e226dfaeb8cdc42f665422522d14,canceled,2016-11-08 10:58:34
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,canceled,2016-11-09 14:53:50
